# Neural Machine Translation with Transformers

This notebook demonstrates how to build an end-to-end neural machine translation pipeline using the Hugging Face Transformers library. It covers dataset preparation, tokenization, preprocessing, evaluation, and fine-tuning of a pretrained sequence-to-sequence Transformer model.

## Learning Objectives

- Load and explore a machine translation dataset
- Understand sequence-to-sequence (Seq2Seq) models
- Tokenize source and target languages
- Prepare translation datasets for training
- Fine-tune pretrained translation models
- Evaluate translation quality using BLEU

## Technologies

- Python
- PyTorch
- Hugging Face Transformers
- Hugging Face Datasets
- Hugging Face Evaluate
- Hugging Face Hub

In [17]:
!git config --global user.email "milad.saeedi@mail.utoronto.ca"
!git config --global user.name "Milad Saeedi"

## Loading the Translation Dataset

The first step is to load a bilingual dataset containing aligned sentence pairs. Each example consists of a source sentence and its corresponding translation in the target language.

In [18]:
from datasets import load_dataset

raw_datasets = load_dataset("kde4", lang1="en", lang2="fr")

### Exploring the Dataset

Before preprocessing, we inspect the dataset structure to understand how the source and target translations are organized.

In [19]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 210173
    })
})

In [20]:
from datasets import DatasetDict
raw_datasets = DatasetDict({
    "train": raw_datasets["train"]
        .shuffle(seed=42)
        .select(range(10000))
})

raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 10000
    })
})

## Preparing the Dataset

The dataset is divided into training and validation subsets. The training set is used to optimize the model parameters, while the validation set measures how well the model generalizes to unseen translation examples.

In [21]:
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 1000
    })
})

In [22]:
split_datasets["validation"] = split_datasets.pop("test")

In [23]:
split_datasets["train"][1]["translation"]

{'en': '& Player:', 'fr': '& Lecteur & #160;:'}

In [24]:
from transformers import pipeline

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
translator = pipeline("translation", model=model_checkpoint)
translator("Default to expanded threads")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/Users/miladsaeedi/miniforge3/envs/huggingface/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use mps:0


[{'translation_text': 'Par défaut pour les threads élargis'}]

In [25]:
split_datasets["train"][172]["translation"]

{'en': 'Ruler: Select starting planet.',
 'fr': 'Distance & #160;: planète de départ.'}

In [26]:
translator(
    "Unable to import %1 using the OFX importer plugin. This file is not the correct format."
)

[{'translation_text': "Impossible d'importer %1 en utilisant le plugin d'importateur OFX. Ce fichier n'est pas le bon format."}]

## Tokenizing Source and Target Sentences

Sequence-to-sequence models require both the source language and the target language to be tokenized. This section demonstrates how the tokenizer prepares each language for training.

In [27]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")

In [28]:
en_sentence = split_datasets["train"][1]["translation"]["en"]
fr_sentence = split_datasets["train"][1]["translation"]["fr"]

inputs = tokenizer(en_sentence, text_target=fr_sentence)
inputs

{'input_ids': [402, 22509, 37, 0], 'attention_mask': [1, 1, 1, 1], 'labels': [402, 60, 15542, 402, 38492, 0]}

### Tokenizing the Target Language

Unlike text classification tasks, translation requires tokenizing both the input sentence and its expected translation. The target tokens serve as labels during supervised training.

In [29]:
wrong_targets = tokenizer(fr_sentence)
print(tokenizer.convert_ids_to_tokens(wrong_targets["input_ids"]))
print(tokenizer.convert_ids_to_tokens(inputs["labels"]))

['▁&', '▁L', 'ect', 'eur', '▁&', '▁#1', '60', ';', ':', '</s>']
['▁&', '▁Le', 'cteur', '▁&', '▁#160;:', '</s>']


In [30]:
max_length = 128


def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(
        inputs, text_target=targets, max_length=max_length, truncation=True
    )
    return model_inputs

### Preprocessing the Dataset

The tokenizer is applied to every sentence pair, producing numerical representations that can be used for sequence-to-sequence model training.

In [31]:
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names,
)

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Loading a Sequence-to-Sequence Model

Machine translation models use an encoder-decoder architecture. The encoder processes the source sentence, while the decoder generates the translated sentence one token at a time.

In [32]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

### Dynamic Padding for Seq2Seq Models

Translation datasets contain sentences with varying lengths. The data collator dynamically pads each batch while ensuring that decoder inputs and labels remain correctly aligned.

In [33]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [34]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[  402, 22509,    37,     0, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513],
        [  443,     4,  3647,   108,    94,   212,  2057,  2346,    18,    96,
         37875,     4,   293,   151,    86,    45,  3765,   479,     3,    35,
          4118,   115,  4031,     4, 13809,     2,   542,    18,   179,    57,
           378,   759,  9570,     2,    57,  3826,    15,   681, 13809,    64,
             4,  1156,  5311,     3,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,


In [35]:
batch["labels"]

tensor([[  402,    60, 15542,   402, 38492,     0,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
        [  350,    19, 20387,    15,    34,  3404, 10084,    31,   113, 33295,
             2,    66,   439,    19,   698,    17,   100,   906,  1588, 11796,
           274,   769,     3,    87,     6,  8543,   168,  7861,     8, 13809,
             2,  1128, 16783,   146,  1737,  9570,     2,    59, 18036,    38,
           495, 13809,    31,     8,   863,  3204,     3,     0]])

In [36]:
batch["decoder_input_ids"]

tensor([[59513,   402,    60, 15542,   402, 38492,     0, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513],
        [59513,   350,    19, 20387,    15,    34,  3404, 10084,    31,   113,
         33295,     2,    66,   439,    19,   698,    17,   100,   906,  1588,
         11796,   274,   769,     3,    87,     6,  8543,   168,  7861,     8,
         13809,     2,  1128, 16783,   146,  1737,  9570,     2,    59, 18036,
            38,   495, 13809,    31,     8,   863,  3204,     3]])

In [37]:
for i in range(1, 3):
    print(tokenized_datasets["train"][i]["labels"])

[402, 60, 15542, 402, 38492, 0]
[350, 19, 20387, 15, 34, 3404, 10084, 31, 113, 33295, 2, 66, 439, 19, 698, 17, 100, 906, 1588, 11796, 274, 769, 3, 87, 6, 8543, 168, 7861, 8, 13809, 2, 1128, 16783, 146, 1737, 9570, 2, 59, 18036, 38, 495, 13809, 31, 8, 863, 3204, 3, 0]


## Evaluating Translation Quality

Machine translation models are commonly evaluated using the BLEU metric, which measures the similarity between generated translations and reference translations.

In this notebook, we use the `sacrebleu` implementation provided through the Hugging Face `evaluate` library.

In [38]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sacrebleu]


In [39]:
import evaluate

metric = evaluate.load("sacrebleu")

In [40]:
predictions = [
    "This plugin lets you translate web pages between several languages automatically."
]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 46.750469682990165,
 'counts': [11, 6, 4, 3],
 'totals': [12, 11, 10, 9],
 'precisions': [91.66666666666667,
  54.54545454545455,
  40.0,
  33.333333333333336],
 'bp': 0.9200444146293233,
 'sys_len': 12,
 'ref_len': 13}

In [41]:
predictions = ["This This This This"]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 1.683602693167689,
 'counts': [1, 0, 0, 0],
 'totals': [4, 3, 2, 1],
 'precisions': [25.0, 16.666666666666668, 12.5, 12.5],
 'bp': 0.10539922456186433,
 'sys_len': 4,
 'ref_len': 13}

In [42]:
predictions = ["This plugin"]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 0.0,
 'counts': [2, 1, 0, 0],
 'totals': [2, 1, 0, 0],
 'precisions': [100.0, 100.0, 0.0, 0.0],
 'bp': 0.004086771438464067,
 'sys_len': 2,
 'ref_len': 13}

In [43]:
import numpy as np


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100s in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

## Configuring Model Training

The `Seq2SeqTrainingArguments` class defines the training configuration, including batch size, learning rate, evaluation strategy, generation parameters, and checkpointing.

In [45]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    f"marian-finetuned-kde4-en-to-fr_translate",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True,
)

In [46]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/var/folders/nn/c8hngzbx03v9v7qk14mjl0680000gn/T/ipykernel_60476/543751272.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.evaluate(max_length=max_length)

/Users/miladsaeedi/miniforge3/envs/huggingface/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
trainer.train()

In [ ]:
trainer.evaluate(max_length=max_length)

{'eval_loss': 0.8558505773544312,
 'eval_bleu': 52.94161337775576,
 'eval_runtime': 714.2576,
 'eval_samples_per_second': 29.426,
 'eval_steps_per_second': 0.461,
 'epoch': 3.0}

In [ ]:
trainer.push_to_hub(tags="translation", commit_message="Training complete")

'https://huggingface.co/sgugger/marian-finetuned-kde4-en-to-fr/commit/3601d621e3baae2bc63d3311452535f8f58f6ef3'

---

# Key Takeaways

In this notebook, I learned how to:

- Load and preprocess bilingual translation datasets.
- Tokenize source and target languages for sequence-to-sequence learning.
- Prepare translation datasets for supervised training.
- Fine-tune pretrained encoder-decoder Transformer models.
- Evaluate translation quality using BLEU.
- Share fine-tuned translation models on the Hugging Face Hub.